# Notebook 02 · Konduri best classical models on our FRED-MD data

**Paper source:** Teja Konduri and Qian Li, *Forecasting Macroeconomic Variables: A Systematic Comparison of Machine Learning Methods* (July 17, 2024). PDF: <https://www.tejakonduri.com/uploads/Konduri_JMP.pdf>

**Task.** Recreate the paper's strongest practical classical model set on our processed capstone data, not just gradient/ensemble-DI models. Based on Konduri & Li's Table 1 model universe and Tables 2-8 results, this notebook runs the following eight model families:

1. **ARD(6)** benchmark
2. **Random Walk** benchmark for S&P 500 returns
3. **AdaBoost**
4. **AdaBoost-DI**
5. **kNN-DI**
6. **SVR linear-DI**
7. **Random Forest**
8. **XGBoost-DI**

**Why these eight?** Tables 2-6 show that real variables are often best forecast by AdaBoost, kNN with diffusion indices, SVR linear with diffusion indices, Random Forest, and XGBoost-DI. Tables 4-5/7-8 show that CPI and financial variables are hard for ML; the ARD or Random Walk baselines are themselves important winners. Therefore the fair capstone baseline is not only gradient boosting: it must include both econometric baselines and the recurrent top ML models from the paper.

**Important methodological note.** This notebook applies those model families to **our data substrate** from `01_data_prep.ipynb`. It is not an exact Konduri replication: our dataset is the March-2026 FRED-MD vintage, our horizons are `{1,3,6,12}`, and our shared folds are expanding 20-year train / 5-year test blocks. Every deviation is documented so the capstone write-up can be transparent.


## 1 · Paper extraction with LiteParse

I downloaded and parsed the paper locally with LiteParse:

```bash
curl -L https://www.tejakonduri.com/uploads/Konduri_JMP.pdf -o literature/Konduri_JMP.pdf
npx -y @llamaindex/liteparse parse literature/Konduri_JMP.pdf --no-ocr -q \
  -o literature/Konduri_JMP_liteparse.md
```

The parsed markdown is stored at `literature/Konduri_JMP_liteparse.md`. The cells below quote the table/model lines used to choose the eight models in this notebook.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PAPER_URL = "https://www.tejakonduri.com/uploads/Konduri_JMP.pdf"
PAPER_MD = PROJECT_ROOT / "literature" / "Konduri_JMP_liteparse.md"
print("Paper URL:", PAPER_URL)
print("LiteParse markdown:", PAPER_MD)
assert PAPER_MD.exists(), "Expected parsed paper markdown in literature/."

paper_text = PAPER_MD.read_text()
paper_lines = paper_text.splitlines()

def show_lines(start, end):
    for i in range(start, end + 1):
        print(f"{i:4d}: {paper_lines[i-1]}")


Paper URL: https://www.tejakonduri.com/uploads/Konduri_JMP.pdf
LiteParse markdown: /Users/manuelarce/Seafile/My Library/UChicago/Spring 2026/capstone/capstone-quantum/literature/Konduri_JMP_liteparse.md


In [2]:
# Forecast design from section 2.3: rolling POOS, horizons, six lags.
show_lines(199, 217)


 199: 147 2.3 Forecasting Methodology
 200: 
 201: 148 2.3.1 Pseudo Out-Of-Sample Forecasting Design
 202: 
 203: 149 For our forecasting exercise, we adopt a pseudo-out-of-sample approach for the period
 204: 150 from January 1970 to December 2019. Our forecast horizons span 1, 3, 6, 9, and 12
 205: 151 months, with 593 − h evaluation periods for each forecasting horizon h. We use rolling
 206: 152 windows for model estimation, with a window size of 120 − h months. For example,
 207: 153 the forecast for January 1970 is based on data from January 1960 to December 1969;
 208: 
 209:         6
 210: 
 211: 154 similarly, the forecast for February 1970 relies on data from February 1960 to January
 212: 155 1970, and so on. This rolling window approach ensures consistency and eases comparison
 213: 156 across different models while dynamically adapting to use the latest available data.
 214: 157 To simplify cross-model comparisons, we uniformly use six lags across all evaluation
 215: 158

In [3]:
# Table 1: full model universe. This is where ARD/RW, AdaBoost, RF, kNN-DI, SVR-DI, and XGBoost-DI come from.
show_lines(1073, 1104)


1073:     Table 1: List of all forecasting models
1074: 
1075:                             Model list
1076: Baseline models
1077: ARD                     Autoregressive direct
1078: RW                      Random walk
1079: Individual machine learning models
1080: kNN(uniform)            K-nearest neighbor (uniform weighted) regression
1081: kNN(inverse)            K-nearest neighbor (inverse weighted) regression
1082: Decision Tree           Decision tree regression
1083: SVR (linear)            Support vector regression (linear kernel)
1084: SVR (polynomial)        Support vector regression (polynomial kernel)
1085: SVR (rbf)               Support vector regression (rbf kernel)
1086: SVR (sigmoid)           Support vector regression (sigmoid kernel)
1087: Ensemble machine learning models
1088: Random forest           Random forest regression
1089: XGBoost                 XGBoost regression
1090: AdaBoost                AdaBoost regression
1091: Gradient Boost          Gradient boost 

In [4]:
# Section 3.4: diffusion indices are PCA factors.
show_lines(410, 424)


 410: 299 3.4 Machine Learning Models Using Dimension Reduction
 411: 
 412: 300 Our dataset comprises over 100 macroeconomic variables. Due to the intricate inter-
 413: 301 dependencies between the predictors and the forecasted variables, there is a high risk
 414: 302 of overfitting. We integrate diffusion indices (DIs) derived from principal component
 415: 303 analysis (PCA) into our forecasting models to address these challenges.
 416: 304 Ma & Zhu (2013) and Kotchoni et al. (2019) identify three key methods to improve
 417: 305 out-of-sample forecasting accuracy while mitigating overfitting: sparse modeling, regu-
 418: 306 larization, and dense modeling. We adopt dense modeling via PCA, which assumes that
 419: 307 a few principal components can significantly capture the variance in the data. These
 420: 308 components, our DIs, condense the dataset’s vast information into a manageable form,
 421: 309 enhancing model efficiency 1. DIs retain the information that has the most pr

In [5]:
# Table 6: best models for real variables, including Industrial Production and Employment.
show_lines(1304, 1340)


1304:     Table 6: Relative RMSPE of the best models for real variables (sample period: 1960m1-2019m12)
1305: 
1306:                              Pre-Pandemic Sample                                         NBER recession periods
1307: Model               h=1      h=3      h=6    h=9   h=12                       h=1     h=3         h=6     h=9      h=12
1308: Industrial Production
1309: Baseline - ARD  0.082       0.061   0.057  0.052  0.047    Baseline - ARD     0.14  0.111        0.101   0.091    0.084
1310: 
1311: kNN (inverse)       0.99     1.00    0.92   0.90   0.93    SVR (linear)       1.00   1.00         0.87   0.83*   0.73***
1312: AdaBoost      0.95***        0.94    1.00   1.00   0.96    AdaBoost           0.94  0.88*         0.99    0.88    0.76**
1313: kNN (uniform) - DI  1.02     0.96   0.87*  0.89*   0.94    Random Forest      0.95   0.89         0.96    0.86    0.78**
1314: kNN (inverse) - DI  1.02     0.96   0.88*  0.89*   0.94    Gradient Boost     1.12   1.02        

In [6]:
# Tables 7-8: CPI and S&P 500. These motivate keeping ARD/RW baselines and AdaBoost as serious contenders.
show_lines(1360, 1372)
print("\n--- S&P 500 excerpt ---")
show_lines(1416, 1428)


1360: Table 7: Relative RMSPE of the best models for nominal variables (sample period: 1960m1-2019m12)
1361: 
1362:                            Pre-Pandemic Sample               NBER recession periods
1363: Model                h=1    h=3    h=6    h=9    h=12                         h=1    h=3    h=6     h=9    h=12
1364: CPI: All Items
1365: Baseline - ARD      0.031  0.027  0.023  0.021  0.021    Baseline - ARD      0.052  0.048  0.036   0.032   0.03
1366: 
1367: kNN (uniform)        1.12   1.27   1.44   1.51   1.52    kNN (uniform)        0.96   1.14   1.59    1.35   1.36
1368: Random Forest        1.12   1.27   1.37   1.33   1.33    Random Forest        1.02   1.32   1.46    1.15   1.20
1369: AdaBoost             1.07   1.20   1.25   1.29   1.31    AdaBoost             0.92   1.20   1.39    1.19   1.20
1370: AdaBoost - DI        1.05   1.14   1.23   1.28   1.29    AdaBoost - DI        0.89   1.01   1.33    1.16   1.17
1371: Gradient Boost - DI  1.32   1.36   1.41   1.62   1.52    G

## 2 · Preprocessing audit of the existing project substrate

This notebook intentionally **does not redo** `01_data_prep.ipynb`; it consumes the saved artifacts so model comparisons remain consistent across the capstone.

Key decisions from the previous notebook:

- FRED-MD March 2026 vintage.
- 126 raw variables; 123 retained after dropping `ACOGNO`, `ANDENOx`, and `TWEXAFEGSMTHx`.
- McCracken-Ng transformations applied to create a stationary panel.
- Shared targets: cumulative log growth for `INDPRO`, `PAYEMS`, `CPIAUCSL`, and `S&P 500` at `h = {1,3,6,12}`.
- Shared expanding folds: 20-year initial training window, 5-year test blocks, one-year step.

### Transparency flags

1. **Missing-value handling.** The previous notebook text says missing values are handled past-only, but the code also uses `bfill(limit=3)`. If this touched interior gaps, it can use future observations. I do not change the substrate here, but this should be audited before final claims.
2. **Konduri target scaling.** Konduri annualizes targets using `1200/h`. Our stored targets are unannualized cumulative log growth. Relative RMSPE is invariant to a horizon-specific scaling, but absolute RMSPE is not directly comparable to the paper.
3. **CPI definition.** Konduri treats log CPI as I(2). Our project target is CPI cumulative log growth over horizon `h`. This is a project target choice, not an exact CPI-target replication.
4. **Fold design.** Konduri uses 120-month rolling windows with monthly forecast origins. Our capstone substrate uses expanding windows with overlapping 5-year test blocks. The notebook deduplicates overlapping test predictions before headline metrics.
5. **Horizon leakage.** For target `y_{t+h}`, training labels in the last `h` months of a training fold would not be observed at the forecast origin. This notebook purges those rows before fitting supervised models.


In [7]:
import json
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

metadata = json.loads((DATA_DIR / "metadata.json").read_text())
print(json.dumps({k: metadata[k] for k in ["vintage", "stationary_range", "dropped_series", "adf_failures", "zero_positive_folds"]}, indent=2))
print("\nArtifacts used:")
for name in ["stationary_panel.parquet", "targets.parquet", "usrec.parquet", "folds.json"]:
    print(f"  {name:26s} {metadata['artifacts'][name].get('shape', '')}  {metadata['artifacts'][name].get('desc', '')}")


{
  "vintage": "2026-03-MD.csv",
  "stationary_range": [
    "1962-04-01",
    "2026-02-01"
  ],
  "dropped_series": [
    "ACOGNO",
    "ANDENOx",
    "TWEXAFEGSMTHx"
  ],
  "adf_failures": [
    "HOUSTMW",
    "HOUSTNE",
    "PERMITMW"
  ],
  "zero_positive_folds": 15
}

Artifacts used:
  stationary_panel.parquet   [767, 123]  Input to Track C/D pipelines (stationarized)
  targets.parquet            [806, 20]  16 regression + 4 classification targets
  usrec.parquet              [806, 1]  NBER USREC monthly 0/1 indicator
  folds.json                   39 expanding-window folds with date ranges


## 3 · Assumptions, decisions, and limitations

This section is intentionally explicit. A grader should be able to tell exactly what is assumed, what comes from Konduri & Li, what comes from our previous preprocessing notebook, and what is a pragmatic capstone choice.

### 3.1 Data assumptions

| Area | Assumption / decision | Why it matters |
|---|---|---|
| Raw source | We use the March 2026 FRED-MD vintage at `data/2026-03-MD.csv`. | Konduri uses a revised FRED-MD-style panel through 2023 with added ISM/YCharts PMI series. Our exact input data are not identical. |
| Frequency | All observations are monthly and indexed by month start. | Forecast origins, horizons, and folds are monthly. |
| Available predictors | We use the 123 retained series from `stationary_panel.parquet`. | Konduri reports 134 monthly macro/financial series after augmenting FRED-MD. We do not have the proprietary ISM/YCharts additions in this notebook. |
| Dropped series | `ACOGNO`, `ANDENOx`, and `TWEXAFEGSMTHx` were dropped in preprocessing because of high missingness or late starts. | This avoids truncating the sample, but means our predictor set differs from the paper. |
| Stationarity | We trust the McCracken-Ng transformation codes applied in `01_data_prep.ipynb`. | Most macro ML comparisons assume stationary features; three retained series still failed ADF in metadata: `HOUSTMW`, `HOUSTNE`, `PERMITMW`. |
| Missing values | We consume the preprocessed substrate as saved. We do not do additional imputation here. | The prior notebook used forward fill and also `bfill(limit=3)`. Backfill can leak future values if it hits interior gaps; this must be audited before final claims. |
| Data revisions | We use latest-vintage data, not real-time vintages. | This is standard in many academic forecasting horse races but is not a true real-time forecasting exercise. |
| Publication lags | We do not model release/calendar delays beyond the prior notebook's simple tail fill. | Some variables would not actually be known at the month-end forecast origin in production. |

### 3.2 Target assumptions

| Area | Assumption / decision | Why it matters |
|---|---|---|
| Regression targets | We forecast `INDPRO`, `PAYEMS`, `CPIAUCSL`, and `S&P 500`. | These match the capstone sponsor targets and the key variables in Konduri Tables 2-5. |
| Horizon set | We evaluate `h = {1, 3, 6, 12}`. | Konduri uses `{1, 3, 6, 9, 12}`. We omit `h=9` because the existing project substrate does. |
| Target formula | We use the existing stored targets: cumulative log growth from `t` to `t+h`. | Konduri annualizes by `1200/h`; relative RMSPE is scale-invariant by horizon, but absolute RMSPEs are not directly comparable. |
| CPI target | We forecast CPI cumulative log growth as stored. | Konduri treats log CPI as I(2) and uses a transformation tied to inflation acceleration. Our CPI results should be described as capstone-target CPI forecasting, not exact Konduri CPI replication. |
| S&P 500 benchmark | Random Walk predicts zero future log return. | This follows the finance convention used in the paper for financial variables. |
| Label timing | For each fold and horizon, we purge the last `h` months of the training window. | Otherwise we would train on labels whose future outcome would not have been known at the forecast origin. |

### 3.3 Fold and evaluation assumptions

| Area | Assumption / decision | Why it matters |
|---|---|---|
| Train/test protocol | We use the shared project folds: 20-year expanding training window, 5-year test block, 12-month step. | Konduri uses 120-month rolling windows with monthly forecast origins. This notebook is a capstone-baseline recreation, not an exact replication. |
| Overlapping tests | Because 5-year test blocks step by one year, test dates overlap across folds. We keep the latest valid forecast for each `(model, target, horizon, date)`. | Without deduplication, repeated dates would be over-weighted in RMSPE. |
| Metric | We use RMSPE = root mean squared prediction error. | This matches the paper's point-forecast metric. |
| Relative RMSPE | For `INDPRO`, `PAYEMS`, and `CPIAUCSL`, relative RMSPE is against `ARD(6)`. For `S&P 500`, it is against `Random Walk`. | This follows the paper's benchmark split between macro variables and financial variables. |
| Recession subset | Recession metrics use rows where `USREC_{t+h}=1`, i.e. the target month is a recession month. | This follows the paper's interpretation that the target observation belongs to a recession episode. Small recession sample sizes mean these results are noisy. |
| Statistical testing | We do not implement Diebold-Mariano tests in this notebook. | Konduri reports DM significance. Our current notebook ranks by RMSPE only. |

### 3.4 Feature-engineering assumptions

| Area | Assumption / decision | Why it matters |
|---|---|---|
| Lags | All supervised models use six lags, `lag0` through `lag5`. | Konduri states that all models use six lags. `lag0` is information known at forecast origin `t`. |
| Full-panel features | `AdaBoost` and `Random Forest` use six lags of the full 123-variable stationary panel. | This gives them access to the same high-dimensional macro information as the DI models, without PCA. |
| Diffusion indices | DI models use PCA factors explaining 80% of training-fold variance. | Konduri chooses factor counts using Bai-Ng panel criteria and selects the smallest recommended number; our 80% rule is a transparent approximation using the existing project design. |
| PCA leakage control | StandardScaler and PCA are fit only on each fold's training months, then applied to test months. | Prevents test-window information from influencing factors. |
| Scaling | kNN-DI and SVR linear-DI include StandardScaler after PCA-lag construction; tree models do not require scaling. | Distance/kernel methods are scale-sensitive. Tree methods are much less scale-sensitive. |

### 3.5 Model and hyperparameter assumptions

Hyperparameters are intentionally simple and fixed rather than tuned on test data.

| Model | Implementation | Main assumptions |
|---|---|---|
| `ARD(6)` | `LinearRegression` on six lags of the target's stationary series | No regularization; direct h-step forecast. |
| `Random Walk` | zero predicted log return for S&P 500 | Appropriate only for return-style financial target. |
| `AdaBoost` | `AdaBoostRegressor` with shallow decision-tree base learner | Fixed `n_estimators=50`, `learning_rate=0.05`, `max_depth=3`. |
| `Random Forest` | `RandomForestRegressor` | Fixed `n_estimators=40`, `min_samples_leaf=3`, `max_features='sqrt'`. |
| `AdaBoost-DI` | same AdaBoost on PCA-lag features | PCA factors refit per fold. |
| `kNN-DI` | `KNeighborsRegressor(n_neighbors=10, weights='distance')` | Konduri discusses `k=10` for 120-month windows; we keep `k=10` for comparability. |
| `SVR linear-DI` | linear `SVR` in a scaled pipeline, target also scaled | Fixed `C=1.0`, `epsilon=0.05`; not tuned. |
| `XGBoost-DI` | `XGBRegressor` | Fixed conservative tree boosting parameters; no test-set tuning. |

### 3.6 Scope limitations

- This notebook is for **transparent benchmark construction**, not final causal/economic interpretation.
- We do not use real-time data vintages, publication lag calendars, or nested hyperparameter tuning.
- We do not yet compare against quantum models; this notebook creates the classical benchmark they must beat.
- The current result tables should be described as: **Konduri-inspired best-model set, run on our processed capstone dataset and folds**.


## 4 · Model design on our data

### Models and feature sets

| Model in notebook | Paper source | Feature set here | Notes |
|---|---|---|---|
| `ARD(6)` | Table 1 baseline; Tables 2-4/6-7 | six lags of target's stationary series | benchmark for non-financial targets |
| `Random Walk` | Table 1 baseline; Tables 5/8 | zero future log return | benchmark for `S&P 500` |
| `AdaBoost` | Table 1; strong in Tables 2, 3, 5, 6, 8 | full 123-variable stationary panel with six lags | tree boosting on raw macro features |
| `Random Forest` | Table 1; strong in Tables 3 and 6 | full 123-variable stationary panel with six lags | bagged trees on raw macro features |
| `AdaBoost-DI` | Table 1; recurring top DI model in Tables 2-8 | PCA diffusion indices with six lags | PCA fit per fold only |
| `kNN-DI` | Table 1; strong in Tables 2 and 6 | PCA diffusion indices with six lags | inverse-distance kNN, following the paper's kNN inverse variant |
| `SVR linear-DI` | Table 1; very strong for employment/recession horizons | PCA diffusion indices with six lags | scaled linear SVR |
| `XGBoost-DI` | Table 1; strong for employment/unemployment style variables | PCA diffusion indices with six lags | gradient boosted trees on PCA factors |

### Leakage controls

- Full-panel lag features use only information available at time `t`: `lag0` through `lag5`.
- PCA is fit on the training fold only, then applied to train/test months.
- The last `h` months of each training fold are purged before fitting because their `t+h` labels would not be known.
- Because shared 5-year test folds overlap, headline metrics keep the latest prediction for each `(model, target, horizon, date)`.


In [8]:
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("XGBoost unavailable:", repr(e))

SEED = 42
LAGS = 6
PCA_VARIANCE = 0.80
DEDUP_POLICY = "latest"

stationary = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
targets = pd.read_parquet(DATA_DIR / "targets.parquet")
usrec = pd.read_parquet(DATA_DIR / "usrec.parquet")["USREC"]

stationary.index = pd.to_datetime(stationary.index)
targets.index = pd.to_datetime(targets.index)
usrec.index = pd.to_datetime(usrec.index)

with open(DATA_DIR / "folds.json") as f:
    folds_json = json.load(f)["shared"]

folds = []
for i, f in enumerate(folds_json):
    train_idx = stationary.loc[f["train_start"]:f["train_end"]].index
    test_idx = stationary.loc[f["test_start"]:f["test_end"]].index
    folds.append({**f, "fold_id": i, "train_idx": train_idx, "test_idx": test_idx})

reg_targets = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
horizons = [1, 3, 6, 12]

def target_col(series, h):
    return f"y_{series}_h{h}"

print("stationary:", stationary.shape, stationary.index.min(), "to", stationary.index.max())
print("targets   :", targets.shape, targets.index.min(), "to", targets.index.max())
print(f"folds     : {len(folds)}")


stationary: (767, 123) 1962-04-01 00:00:00 to 2026-02-01 00:00:00
targets   : (806, 20) 1959-01-01 00:00:00 to 2026-02-01 00:00:00
folds     : 39


In [9]:
def make_lagged(df: pd.DataFrame, lags: int) -> pd.DataFrame:
    '''Return lag0..lag(lags-1), where lag0 is information known at forecast origin t.'''
    return pd.concat([df.shift(lag).add_suffix(f"_lag{lag}") for lag in range(lags)], axis=1)


def rmspe(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))


def full_model_factories():
    '''Models selected from Konduri tables that use the full macro panel here.'''
    return {
        "AdaBoost": AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=SEED),
            n_estimators=50,
            learning_rate=0.05,
            random_state=SEED,
        ),
        "Random Forest": RandomForestRegressor(
            n_estimators=40,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=SEED,
            n_jobs=-1,
        ),
    }


def di_model_factories():
    '''Models selected from Konduri tables that use PCA diffusion-index features here.'''
    models = {
        "AdaBoost-DI": AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=SEED),
            n_estimators=50,
            learning_rate=0.05,
            random_state=SEED,
        ),
        "kNN-DI": Pipeline([
            ("scale", StandardScaler()),
            ("knn", KNeighborsRegressor(n_neighbors=10, weights="distance", metric="euclidean")),
        ]),
        "SVR linear-DI": TransformedTargetRegressor(
            regressor=Pipeline([
                ("scale", StandardScaler()),
                ("svr", SVR(kernel="linear", C=1.0, epsilon=0.05)),
            ]),
            transformer=StandardScaler(),
        ),
    }
    if HAS_XGB:
        models["XGBoost-DI"] = XGBRegressor(
            n_estimators=50,
            learning_rate=0.05,
            max_depth=3,
            min_child_weight=5,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=SEED,
            n_jobs=1,
            verbosity=0,
        )
    return models

MODEL_ORDER = ["ARD(6)", "Random Walk", "AdaBoost", "Random Forest", "AdaBoost-DI", "kNN-DI", "SVR linear-DI", "XGBoost-DI"]
print("Full-panel models:", list(full_model_factories()))
print("DI models        :", list(di_model_factories()))
print("Eight requested model families:", MODEL_ORDER)


Full-panel models: ['AdaBoost', 'Random Forest']
DI models        : ['AdaBoost-DI', 'kNN-DI', 'SVR linear-DI', 'XGBoost-DI']
Eight requested model families: ['ARD(6)', 'Random Walk', 'AdaBoost', 'Random Forest', 'AdaBoost-DI', 'kNN-DI', 'SVR linear-DI', 'XGBoost-DI']


## 4 · Precompute leakage-safe fold features

This step caches the feature matrices so the modeling loop does not refit PCA repeatedly for every target and horizon.


In [10]:
X_full_lagged = make_lagged(stationary, LAGS)

fold_feature_cache = {}
for fold in folds:
    fold_id = fold["fold_id"]
    pca_pipe = Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=PCA_VARIANCE, svd_solver="full", random_state=SEED)),
    ])
    pca_pipe.fit(stationary.loc[fold["train_idx"]])
    factors = pd.DataFrame(
        pca_pipe.transform(stationary),
        index=stationary.index,
        columns=[f"DI{i+1:02d}" for i in range(pca_pipe.named_steps["pca"].n_components_)],
    )
    fold_feature_cache[fold_id] = {
        "X_di_lagged": make_lagged(factors, LAGS),
        "n_di": int(pca_pipe.named_steps["pca"].n_components_),
        "pca_var": float(pca_pipe.named_steps["pca"].explained_variance_ratio_.sum()),
    }

pd.DataFrame([
    {"fold_id": k, "n_di": v["n_di"], "pca_var": v["pca_var"]}
    for k, v in fold_feature_cache.items()
]).describe().round(3)


,fold_id,n_di,pca_var
count,39.000,39.000,39.000
mean,19.000,32.103,0.804
std,11.402,1.353,0.002
min,0.000,29.000,0.800
25%,9.500,31.000,0.803
50%,19.000,33.000,0.804
75%,28.500,33.000,0.805
max,38.000,34.000,0.808


In [11]:
def append_prediction_rows(rows, model_name, series, h, fold, dates, y_true, y_pred, feature_set, extra=None):
    rec = usrec.shift(-h).reindex(dates)
    extra = extra or {}
    for dt, yt, yp in zip(dates, y_true, y_pred):
        rows.append({
            "model": model_name,
            "target": series,
            "h": h,
            "fold_id": fold["fold_id"],
            "fold_train_end": fold["train_end"],
            "date": dt,
            "y_true": float(yt),
            "y_pred": float(yp),
            "feature_set": feature_set,
            "recession_target_month": int(rec.loc[dt]) if pd.notna(rec.loc[dt]) else np.nan,
            **extra,
        })


def valid_xy(X, y, idx):
    Xb = X.reindex(idx)
    yb = y.reindex(idx)
    mask = Xb.notna().all(axis=1) & yb.notna()
    good_idx = idx[mask.values]
    return good_idx, X.loc[good_idx], y.loc[good_idx]


def fit_predict_one(series: str, h: int, fold: dict) -> list[dict]:
    y = targets[target_col(series, h)]
    train_idx = fold["train_idx"]
    test_idx = fold["test_idx"]
    train_end = pd.Timestamp(fold["train_end"])
    supervised_train_idx = train_idx[train_idx <= train_end - pd.DateOffset(months=h)]
    test_eval_idx = y.reindex(test_idx).dropna().index
    rows = []

    # 1) ARD benchmark for non-financial targets.
    if series != "S&P 500":
        X_ard = make_lagged(stationary[[series]], LAGS)
        tr_idx, Xtr, ytr = valid_xy(X_ard, y, supervised_train_idx)
        te_idx, Xte, yte = valid_xy(X_ard, y, test_eval_idx)
        if len(tr_idx) > 10 and len(te_idx) > 0:
            model = LinearRegression()
            model.fit(Xtr, ytr)
            append_prediction_rows(rows, "ARD(6)", series, h, fold, te_idx, yte, model.predict(Xte), "target_lags")

    # 2) Random walk benchmark for S&P 500 log returns.
    if series == "S&P 500":
        y_test = y.reindex(test_eval_idx).dropna()
        append_prediction_rows(rows, "Random Walk", series, h, fold, y_test.index, y_test, np.zeros(len(y_test)), "rw_zero_return")

    # 3) Full-panel ML models.
    tr_idx, Xtr, ytr = valid_xy(X_full_lagged, y, supervised_train_idx)
    te_idx, Xte, yte = valid_xy(X_full_lagged, y, test_eval_idx)
    if len(tr_idx) > 20 and len(te_idx) > 0:
        for name, estimator in full_model_factories().items():
            model = clone(estimator)
            model.fit(Xtr, ytr)
            append_prediction_rows(
                rows, name, series, h, fold, te_idx, yte, model.predict(Xte), "full_panel_lags",
                {"n_train_supervised": int(len(tr_idx)), "n_features": int(Xtr.shape[1])},
            )

    # 4) PCA diffusion-index ML models.
    cache = fold_feature_cache[fold["fold_id"]]
    X_di = cache["X_di_lagged"]
    tr_idx, Xtr, ytr = valid_xy(X_di, y, supervised_train_idx)
    te_idx, Xte, yte = valid_xy(X_di, y, test_eval_idx)
    if len(tr_idx) > 20 and len(te_idx) > 0:
        for name, estimator in di_model_factories().items():
            model = clone(estimator)
            model.fit(Xtr, ytr)
            append_prediction_rows(
                rows, name, series, h, fold, te_idx, yte, model.predict(Xte), "di_pca_lags",
                {
                    "n_train_supervised": int(len(tr_idx)),
                    "n_features": int(Xtr.shape[1]),
                    "n_di": cache["n_di"],
                    "pca_var": cache["pca_var"],
                },
            )

    return rows


## 5 · Run pseudo-out-of-sample forecasts

The cell checkpoints after each target/horizon block. If interrupted, rerun the notebook and it resumes from `results/konduri_best_models_predictions_raw.parquet`.


In [12]:
start = time.time()
checkpoint = RESULTS_DIR / "konduri_best_models_predictions_raw.parquet"

if checkpoint.exists():
    preds_raw = pd.read_parquet(checkpoint)
    done = set(map(tuple, preds_raw[["target", "h"]].drop_duplicates().to_numpy()))
    rows = preds_raw.to_dict("records")
    print(f"Loaded checkpoint with {len(preds_raw):,} rows and {len(done)} completed target/horizon blocks")
else:
    done = set()
    rows = []

total_blocks = len(reg_targets) * len(horizons)
for series in reg_targets:
    for h in horizons:
        if (series, h) in done:
            print(f"skip existing {series:8s} h={h:2d}")
            continue
        block_rows = []
        for fold in folds:
            block_rows.extend(fit_predict_one(series, h, fold))
        rows.extend(block_rows)
        preds_raw = pd.DataFrame(rows)
        preds_raw["date"] = pd.to_datetime(preds_raw["date"])
        preds_raw.to_parquet(checkpoint, index=False)
        done.add((series, h))
        print(f"finished {series:8s} h={h:2d}  block_rows={len(block_rows):,}  completed={len(done)}/{total_blocks}")

preds_raw = pd.DataFrame(rows)
preds_raw["date"] = pd.to_datetime(preds_raw["date"])
preds_raw.to_parquet(checkpoint, index=False)
print(f"Raw prediction rows: {len(preds_raw):,}")
print(f"Elapsed this run: {(time.time() - start)/60:.1f} minutes")
preds_raw.head()


Loaded checkpoint with 262,052 rows and 16 completed target/horizon blocks
skip existing INDPRO   h= 1
skip existing INDPRO   h= 3
skip existing INDPRO   h= 6
skip existing INDPRO   h=12
skip existing PAYEMS   h= 1
skip existing PAYEMS   h= 3
skip existing PAYEMS   h= 6
skip existing PAYEMS   h=12
skip existing CPIAUCSL h= 1
skip existing CPIAUCSL h= 3
skip existing CPIAUCSL h= 6
skip existing CPIAUCSL h=12
skip existing S&P 500  h= 1
skip existing S&P 500  h= 3
skip existing S&P 500  h= 6
skip existing S&P 500  h=12


Raw prediction rows: 262,052
Elapsed this run: 0.0 minutes


,model,target,h,fold_id,fold_train_end,date,y_true,y_pred,feature_set,recession_target_month,n_train_supervised,n_features,n_di,pca_var
0,ARD(6),INDPRO,1,0,1982-03-01,1982-04-01,-0.006679,-0.000367,target_lags,1,NaN,NaN,NaN,NaN
1,ARD(6),INDPRO,1,0,1982-03-01,1982-05-01,-0.002607,0.001661,target_lags,1,NaN,NaN,NaN,NaN
2,ARD(6),INDPRO,1,0,1982-03-01,1982-06-01,-0.003292,-0.004521,target_lags,1,NaN,NaN,NaN,NaN
3,ARD(6),INDPRO,1,0,1982-03-01,1982-07-01,-0.009171,-0.000111,target_lags,1,NaN,NaN,NaN,NaN
4,ARD(6),INDPRO,1,0,1982-03-01,1982-08-01,-0.002469,-0.001671,target_lags,1,NaN,NaN,NaN,NaN


In [13]:
# Deduplicate overlapping test windows: keep latest fold's prediction for each model/target/h/date.
preds = preds_raw.copy()
preds["fold_train_end"] = pd.to_datetime(preds["fold_train_end"])
preds = preds.sort_values(["model", "target", "h", "date", "fold_train_end"])
if DEDUP_POLICY == "latest":
    preds = preds.drop_duplicates(["model", "target", "h", "date"], keep="last")
elif DEDUP_POLICY == "earliest":
    preds = preds.drop_duplicates(["model", "target", "h", "date"], keep="first")
else:
    raise ValueError(DEDUP_POLICY)

preds.to_parquet(RESULTS_DIR / "konduri_best_models_predictions.parquet", index=False)
print(f"Deduplicated prediction rows: {len(preds):,}")
print(preds.groupby("model").size().reindex(MODEL_ORDER).dropna().astype(int))


Deduplicated prediction rows: 57,764


model
ARD(6)           6189
Random Walk      2063
AdaBoost         8252
Random Forest    8252
AdaBoost-DI      8252
kNN-DI           8252
SVR linear-DI    8252
XGBoost-DI       8252
dtype: int64


## 6 · Results on our data

For relative RMSPE, the benchmark is `ARD(6)` for `INDPRO`, `PAYEMS`, and `CPIAUCSL`, and `Random Walk` for `S&P 500`. Values below 1.00 beat the relevant benchmark.


In [14]:
def benchmark_name(target):
    return "Random Walk" if target == "S&P 500" else "ARD(6)"


def summarize_metrics(pred_df: pd.DataFrame, recession_only: bool = False) -> pd.DataFrame:
    df = pred_df.copy()
    if recession_only:
        df = df[df["recession_target_month"] == 1]
    recs = []
    for (target, h), block in df.groupby(["target", "h"]):
        bname = benchmark_name(target)
        base = block[block["model"] == bname]
        if len(base) == 0:
            continue
        base_r = rmspe(base["y_true"], base["y_pred"])
        for model, mb in block.groupby("model"):
            r = rmspe(mb["y_true"], mb["y_pred"])
            recs.append({
                "target": target,
                "h": h,
                "model": model,
                "benchmark": bname,
                "n": len(mb),
                "rmspe": r,
                "benchmark_rmspe": base_r,
                "relative_rmspe": r / base_r if base_r > 0 else np.nan,
            })
    return pd.DataFrame(recs).sort_values(["target", "h", "relative_rmspe"])

metrics = summarize_metrics(preds, recession_only=False)
metrics_recession = summarize_metrics(preds, recession_only=True)
metrics.to_csv(RESULTS_DIR / "konduri_best_models_metrics.csv", index=False)
metrics_recession.to_csv(RESULTS_DIR / "konduri_best_models_metrics_recession.csv", index=False)

rel_table = metrics.pivot_table(index=["target", "model"], columns="h", values="relative_rmspe")
rel_table = rel_table.reindex(columns=horizons)
rel_table.style.format("{:.3f}").background_gradient(axis=None, cmap="RdYlGn_r", vmin=0.7, vmax=1.3)


In [15]:
# Best non-benchmark model by target/horizon.
benchmarks = {"ARD(6)", "Random Walk"}
best = metrics[~metrics["model"].isin(benchmarks)].sort_values("relative_rmspe").groupby(["target", "h"]).head(1)
best = best.sort_values(["target", "h"])
best[["target", "h", "model", "benchmark", "n", "rmspe", "benchmark_rmspe", "relative_rmspe"]].style.format({
    "rmspe": "{:.5f}",
    "benchmark_rmspe": "{:.5f}",
    "relative_rmspe": "{:.3f}",
})


,target,h,model,benchmark,n,rmspe,benchmark_rmspe,relative_rmspe
6,CPIAUCSL,1,kNN-DI,ARD(6),516,0.00263,0.00292,0.903
13,CPIAUCSL,3,kNN-DI,ARD(6),516,0.00604,0.00740,0.817
20,CPIAUCSL,6,kNN-DI,ARD(6),516,0.00999,0.01300,0.769
27,CPIAUCSL,12,kNN-DI,ARD(6),515,0.01771,0.02401,0.738
33,INDPRO,1,XGBoost-DI,ARD(6),516,0.00942,0.01003,0.940
40,INDPRO,3,XGBoost-DI,ARD(6),516,0.01874,0.02044,0.917
47,INDPRO,6,XGBoost-DI,ARD(6),516,0.02806,0.02870,0.978
52,INDPRO,12,Random Forest,ARD(6),515,0.04222,0.04444,0.950
61,PAYEMS,1,XGBoost-DI,ARD(6),516,0.00680,0.00778,0.874
66,PAYEMS,3,Random Forest,ARD(6),516,0.01132,0.01583,0.715


In [16]:
# Recession-target-month subset. Interpret cautiously because some target/horizon cells have few recession observations.
rel_rec = metrics_recession.pivot_table(index=["target", "model"], columns="h", values="relative_rmspe")
rel_rec = rel_rec.reindex(columns=horizons)
rel_rec.style.format("{:.3f}").background_gradient(axis=None, cmap="RdYlGn_r", vmin=0.6, vmax=1.4)


In [17]:
# Compact summary across target/horizon cells where each model is applicable.
summary = (
    metrics[~metrics["model"].isin({"ARD(6)", "Random Walk"})]
    .assign(beats_benchmark=lambda d: d["relative_rmspe"] < 1)
    .groupby("model")
    .agg(
        cells=("relative_rmspe", "size"),
        beats=("beats_benchmark", "sum"),
        mean_relative_rmspe=("relative_rmspe", "mean"),
        median_relative_rmspe=("relative_rmspe", "median"),
        best_relative_rmspe=("relative_rmspe", "min"),
    )
    .sort_values(["beats", "median_relative_rmspe"], ascending=[False, True])
)
summary.style.format({
    "mean_relative_rmspe": "{:.3f}",
    "median_relative_rmspe": "{:.3f}",
    "best_relative_rmspe": "{:.3f}",
})


,cells,beats,mean_relative_rmspe,median_relative_rmspe,best_relative_rmspe
model,,,,,
Random Forest,16,15,0.875,0.921,0.562
XGBoost-DI,16,14,0.868,0.915,0.584
AdaBoost-DI,16,12,0.873,0.905,0.608
AdaBoost,16,12,0.896,0.938,0.608
kNN-DI,16,10,0.913,0.944,0.665
SVR linear-DI,16,4,1.281,1.310,0.792


## 7 · Notes for the capstone write-up

1. **Paper link and table provenance.** The model list comes from Konduri & Li Table 1. The priority set is justified by Tables 2-6 for Industrial Production, Employment, and other real variables; Table 7 for CPI/nominal variables; and Table 8 for S&P 500/financial variables. Paper URL: <https://www.tejakonduri.com/uploads/Konduri_JMP.pdf>.
2. **What we implemented.** ARD(6), Random Walk, AdaBoost, Random Forest, AdaBoost-DI, kNN-DI, SVR linear-DI, and XGBoost-DI.
3. **What is not exact replication.** We use our March-2026 processed data, our project target definitions, horizons `{1,3,6,12}`, and our expanding folds rather than Konduri's 120-month rolling monthly POOS and horizons `{1,3,6,9,12}`.
4. **Leakage controls.** PCA is refit inside each fold; training rows whose `t+h` label would not yet be known are purged; overlapping test-window predictions are deduplicated before headline metrics.
5. **Preprocessing caveat.** The prior notebook's `bfill(limit=3)` should be audited. If it affects interior gaps, it introduces future information into the feature substrate.
6. **Next step for closer replication.** Add a second evaluation mode with Konduri-style 120-month rolling windows and annualized target transformations, then compare whether model rankings are stable.
